# Vibe Coding: Real-World Data Cleaning Challenge

## The Mission

You're a Data Analyst at **TechSalary Insights**. Your manager needs answers to critical business questions, but the data is messy. Your job is to clean it and provide accurate insights.

**The catch:** You must figure out how to clean the data yourself. No step by step hints just you, your AI assistant, and real world messy data.

---

## The Dataset: Ask A Manager Salary Survey 2021

**Location:** `../Week-02-Pandas-Part-2-and-DS-Overview/data/Ask A Manager Salary Survey 2021 (Responses) - Form Responses 1.tsv`

This is **real survey data** from Ask A Manager's 2021 salary survey with over 28,000 responses from working professionals. The data comes from this survey: https://www.askamanager.org/2021/04/how-much-money-do-you-make-4.html

**Why this dataset is perfect for vibe coding:**
- Real human responses (inconsistent formatting)
- Multiple currencies and formats  
- Messy job titles and location data
- Missing and invalid entries
- Requires business judgment calls

---

## Your Business Questions

Answer these **exact questions** with clean data. There's only one correct answer for each:

### Core Questions (Required):
1. **What is the median salary for Software Engineers in the United States?** 
2. **Which US state has the highest average salary for tech workers?**
3. **How much does salary increase on average for each year of experience in tech?**
4. **Which industry (besides tech) has the highest median salary?**

### Bonus Questions (If time permits):
5. **What's the salary gap between men and women in tech roles?**
6. **Do people with Master's degrees earn significantly more than those with Bachelor's degrees?**

**Success Criteria:** Your final answers will be compared against the "official" results. Data cleaning approaches can vary, but final numbers should be within 5% of expected values.


---
# Your Work Starts Here

## Step 0: Create Your Plan
**Before writing any code, use Cursor to create your todo plan. Then paste it here:**

## My Data Cleaning Plan

[▶] Set up workspace and load the TSV dataset into a Pandas DataFrame
* Inspect schema: preview rows, column names, dtypes, nulls, unique values for key fields
* Standardize column names (snake_case), trim whitespace, and normalize text casing
* Parse and clean salary fields: extract numeric amounts, normalize ranges, handle annualization, and remove non-annual entries
* Currency handling: detect currency, convert to USD using survey-year FX rates, flag unconvertible entries
* Normalize job titles: canonicalize to major roles; map variants of Software Engineer
* Location cleaning: extract U.S. state from city/state/country fields; standardize to USPS codes; filter U.S. entries
* Experience normalization: convert years experience and years at employer to numeric, impute/clip invalids
* Define and filter 'tech workers' cohort (industry, job family, or keywords)
* Outlier detection and winsorization rules for salary by role/experience (e.g., remove top/bottom 1% or robust IQR)
* Compute core metrics:
   * median US Software Engineer salary
   * highest average salary by US state for tech workers
   * salary increase per YOE
   * highest median salary industry (ex-tech)
* Validation and sensitivity checks (5% tolerance): compare with alternate cleaning choices
* Document assumptions and cleaning rules for reproducibility
* Produce final answers and brief reasoning summary.



## Step 1: Data Loading and Exploration

Start here! Load the dataset and get familiar with what you're working with.


In [14]:
# Import necessary libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Display all columns in outputs
pd.set_option('display.max_columns', None)

# Load dataset
file_path = "/workspaces/ds-fall-2025-wed-M.E.P./Week-02-Pandas-Part-2-and-DS-Overview/data/Ask A Manager Salary Survey 2021 (Responses) - Form Responses 1.tsv"
df = pd.read_csv(file_path, sep='\t')

In [15]:
# Basic info
print("DataFrame shape:", df.shape)
print("\nColumn names:\n", df.columns.tolist())

# Preview a few rows
display(df.head(3))

DataFrame shape: (28062, 18)

Column names:
 ['Timestamp', 'How old are you?', 'What industry do you work in?', 'Job title', 'If your job title needs additional context, please clarify here:', "What is your annual salary? (You'll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)", 'How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits.', 'Please indicate the currency', 'If "Other," please indicate the currency here: ', 'If your income needs additional context, please provide it here:', 'What country do you work in?', "If you're in the U.S., what state do you work in?", 'What city do you work in?', 'How many years of professional work experience do you have overall?', 'How many years of professional work experien

,Timestamp,How old are you?,What industry do you work in?,Job title,"If your job title needs additional context, please clarify here:","What is your annual salary? (You'll indicate the currency in a later question. If you are part-time or hourly, please enter an annualized equivalent -- what you would earn if you worked the job 40 hours a week, 52 weeks a year.)","How much additional monetary compensation do you get, if any (for example, bonuses or overtime in an average year)? Please only include monetary compensation here, not the value of benefits.",Please indicate the currency,"If ""Other,"" please indicate the currency here:","If your income needs additional context, please provide it here:",What country do you work in?,"If you're in the U.S., what state do you work in?",What city do you work in?,How many years of professional work experience do you have overall?,How many years of professional work experience do you have in your field?,What is your highest level of education completed?,What is your gender?,What is your race? (Choose all that apply.)
0,4/27/2021 11:02:10,25-34,Education (Higher Education),Research and Instruction Librarian,NaN,"55,000",0.0,USD,NaN,NaN,United States,Massachusetts,Boston,5-7 years,5-7 years,Master's degree,Woman,White
1,4/27/2021 11:02:22,25-34,Computing or Tech,Change & Internal Communications Manager,NaN,"54,600",4000.0,GBP,NaN,NaN,United Kingdom,NaN,Cambridge,8 - 10 years,5-7 years,College degree,Non-binary,White
2,4/27/2021 11:02:38,25-34,"Accounting, Banking & Finance",Marketing Specialist,NaN,"34,000",NaN,USD,NaN,NaN,US,Tennessee,Chattanooga,2 - 4 years,2 - 4 years,College degree,Woman,White


In [3]:
# Check data types and null values
print("\n--- Data Info ---")
print(df.info())

print("\n--- Missing Values ---")
print(df.isna().sum().sort_values(ascending=False).head(10))

# Quick look at unique entries for key columns
key_cols = ['How old are you?', 'What industry do you work in?', 'Job title', 'Annual salary', 'Currency', 'Country', 'State', 'Years of experience']
for col in key_cols:
    if col in df.columns:
        print(f"\nUnique values in '{col}': {df[col].nunique()}")


--- Data Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 28062 entries, 0 to 28061
Data columns (total 18 columns):
 #   Column                                                                                                                                                                                                                                Non-Null Count  Dtype  
---  ------                                                                                                                                                                                                                                --------------  -----  
 0   Timestamp                                                                                                                                                                                                                             28062 non-null  object 
 1   How old are you?                                                                          

## Step 2: Data Cleaning


In [4]:
# Make column names snake_case and clean spaces
df.columns = (
    df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
    .str.replace('[^0-9a-zA-Z_]', '', regex=True)
)

# Preview cleaned column names
print("Cleaned columns:\n", df.columns.tolist())


Cleaned columns:
 ['timestamp', 'how_old_are_you', 'what_industry_do_you_work_in', 'job_title', 'if_your_job_title_needs_additional_context_please_clarify_here', 'what_is_your_annual_salary_youll_indicate_the_currency_in_a_later_question_if_you_are_parttime_or_hourly_please_enter_an_annualized_equivalent__what_you_would_earn_if_you_worked_the_job_40_hours_a_week_52_weeks_a_year', 'how_much_additional_monetary_compensation_do_you_get_if_any_for_example_bonuses_or_overtime_in_an_average_year_please_only_include_monetary_compensation_here_not_the_value_of_benefits', 'please_indicate_the_currency', 'if_other_please_indicate_the_currency_here', 'if_your_income_needs_additional_context_please_provide_it_here', 'what_country_do_you_work_in', 'if_youre_in_the_us_what_state_do_you_work_in', 'what_city_do_you_work_in', 'how_many_years_of_professional_work_experience_do_you_have_overall', 'how_many_years_of_professional_work_experience_do_you_have_in_your_field', 'what_is_your_highest_level_of_ed

In [5]:
# Remove completely empty rows
df = df.dropna(how='all')

# --- Clean Salary Field ---
salary_col = [c for c in df.columns if 'salary' in c][0]
print(f"💰 Using salary column: {salary_col}")
df = df[df[salary_col].notna()]

# Convert salary ranges or text to numeric
df[salary_col] = (
    df[salary_col]
    .astype(str)
    .str.replace(',', '', regex=False)
    .str.replace(r'[^0-9\.\-\–]', '', regex=True)
)

💰 Using salary column: what_is_your_annual_salary_youll_indicate_the_currency_in_a_later_question_if_you_are_parttime_or_hourly_please_enter_an_annualized_equivalent__what_you_would_earn_if_you_worked_the_job_40_hours_a_week_52_weeks_a_year


In [6]:
# Extract numeric part (if range, take average)
def parse_salary(value):
    if '-' in value or '–' in value:
        parts = re.split(r'[-–]', value)
        try:
            nums = [float(p) for p in parts if p.strip()]
            return np.mean(nums)
        except:
            return np.nan
    else:
        try:
            return float(value)
        except:
            return np.nan
        
import re
df['salary_num'] = df[salary_col].apply(parse_salary)

# Remove unrealistic values
df = df[(df['salary_num'] >= 10000) & (df['salary_num'] <= 1_000_000)]

# 🪙 3. Handle currency (if exists)
if 'currency' in df.columns:
    df['currency'] = df['currency'].str.upper().str.strip()
    fx_rates = {'USD': 1, 'CAD': 0.79, 'EUR': 1.18, 'GBP': 1.39, 'AUD': 0.74}
    df['salary_usd'] = df.apply(lambda x: x['salary_num'] * fx_rates.get(x['currency'], np.nan), axis=1)
else:
    df['salary_usd'] = df['salary_num']

# 💼 4. Clean and simplify job titles
job_col = [c for c in df.columns if 'job' in c and 'title' in c]
if job_col:
    job_col = job_col[0]
    print(f"🧑‍💻 Job title column: {job_col}")
    df['job_title_clean'] = (
        df[job_col]
        .astype(str)
        .str.strip()
        .str.lower()
        .replace({
            'software developer': 'software engineer',
            'software dev': 'software engineer',
            'backend engineer': 'software engineer',
            'frontend engineer': 'software engineer',
            'full stack engineer': 'software engineer'
        })
    )
else:
    print("⚠️ No job title column found!")
    df['job_title_clean'] = np.nan

# 🌎 5. Detect and clean location/country data
loc_col = None
for c in df.columns:
    if any(x in c for x in ['country', 'location', 'state']):
        loc_col = c
        break

print("📍 Detected location column:", loc_col)

if loc_col:
    df[loc_col] = df[loc_col].astype(str).str.strip()
    df_us = df[df[loc_col].str.contains('United States', case=False, na=False)].copy()
else:
    print("⚠️ No location/country column found.")
    df_us = df.copy()

🧑‍💻 Job title column: job_title
📍 Detected location column: what_country_do_you_work_in


In [7]:
# 📊 6. Normalize experience columns
exp_cols = [c for c in df.columns if 'experience' in c]
for col in exp_cols:
    df_us[col] = pd.to_numeric(df_us[col], errors='coerce')
    df_us[col] = df_us[col].clip(lower=0, upper=50)

# 💻 7. Define tech workers (by keywords)
tech_keywords = ['tech', 'software', 'developer', 'engineer', 'it', 'data', 'computer']
df_us['is_tech'] = df_us['job_title_clean'].apply(lambda x: any(k in str(x) for k in tech_keywords))

# 📉 8. Handle outliers (remove top/bottom 1%)
low, high = df_us['salary_usd'].quantile([0.01, 0.99])
df_us = df_us[(df_us['salary_usd'] >= low) & (df_us['salary_usd'] <= high)]

print("✅ Cleaned dataset shape:", df_us.shape)
print(df_us[['salary_usd', 'job_title_clean', loc_col]].head())


✅ Cleaned dataset shape: (10281, 22)
    salary_usd                     job_title_clean what_country_do_you_work_in
0      55000.0  research and instruction librarian               United States
7      50000.0                           librarian               United States
9      45000.0                   senior accountant               United States
10     47500.0                      office manager               United States
12    100000.0     manager of information services               United States


## Step 3: Business Questions Analysis

Now answer those important business questions!


In [8]:
# Question 1: What is the median salary for Software Engineers in the United States?
se_df = df_us[df_us['job_title_clean'] == 'software engineer']
median_se_salary = se_df['salary_usd'].median()
print(f"1️⃣ Median salary for Software Engineers in US: ${median_se_salary:,.0f}")


1️⃣ Median salary for Software Engineers in US: $120,000


In [9]:
# Question 2: Which US state has the highest average salary for tech workers?
state_avg = df_us[df_us['is_tech']].groupby('if_youre_in_the_us_what_state_do_you_work_in')['salary_usd'].mean().sort_values(ascending=False)
highest_state = state_avg.index[0]
highest_state_salary = state_avg.iloc[0]

print(f"Highest state: {highest_state}")
print(f"Average salary: ${highest_state_salary:,.2f}")


Highest state: California, Colorado
Average salary: $176,000.00


In [10]:
# Question 3: How much does salary increase on average for each year of experience in tech?
if 'years_of_experience' in df_us.columns:
    exp_salary = df_us[df_us['is_tech']][['years_of_experience', 'salary_usd']].dropna()
    coef = np.polyfit(exp_salary['years_of_experience'], exp_salary['salary_usd'], 1)[0]
    print(f"3️⃣ Average salary increase per year of experience: ${coef:,.0f} per year")
else:
    print("3️⃣ Experience data not available.")


3️⃣ Experience data not available.


In [11]:

# Question 4: What percentage of respondents work remotely vs. in-office?
remote_col = [c for c in df_us.columns if 'remote' in c]
if remote_col:
    remote_counts = df_us[remote_col[0]].value_counts(normalize=True) * 100
    print(f"4️⃣ Remote vs Office: {remote_counts.to_dict()}")

In [12]:
# Question 5: Which industry (besides tech) has the highest median salary?
if 'what_industry_do_you_work_in' in df_us.columns:
    non_tech = df_us[~df_us['what_industry_do_you_work_in'].str.contains('tech', case=False, na=False)]
    industry_salary = non_tech.groupby('what_industry_do_you_work_in')['salary_usd'].median().sort_values(ascending=False)
    top_industry = industry_salary.index[0]
    top_industry_salary = industry_salary.iloc[0]
    print(f"5️⃣ Highest paying non-tech industry: {top_industry} (${top_industry_salary:,.0f})")

5️⃣ Highest paying non-tech industry: Pharmaceutical Development ($230,000)


In [16]:
# Bonus Questions:
# Question 6: What's the salary gap between men and women in similar roles?
if 'gender' in df_us.columns:
    gender_salary = df_us.groupby('gender')['salary_usd'].median().dropna()
    print("\n6️⃣ Median salary by gender:")
    print(gender_salary)

# Question 7: Do people with Master's degrees earn significantly more than those with Bachelor's degrees?
edu_cols = [c for c in df_us.columns if 'education' in c]
if edu_cols:
    edu_salary = df_us.groupby(edu_cols[0])['salary_usd'].median().sort_values(ascending=False)
    print("\n7️⃣ Median salary by education level:")
    print(edu_salary.head())

# Question 8: Which company size (startup, medium, large) pays the most on average?
size_cols = [c for c in df_us.columns if 'company_size' in c]
if size_cols:
    size_salary = df_us.groupby(size_cols[0])['salary_usd'].mean().sort_values(ascending=False)
    print("\n8️⃣ Average salary by company size:")
    print(size_salary)




7️⃣ Median salary by education level:
what_is_your_highest_level_of_education_completed
Professional degree (MD, JD, etc.)    114500.0
PhD                                   100800.0
Master's degree                        78000.0
College degree                         75000.0
Some college                           63000.0
Name: salary_usd, dtype: float64


## Final Summary

**Summarize your findings here:**

1. **Median salary for Software Engineers in US:** $120.000
2. **Highest paying US state for tech:** California
3. **Salary increase per year of experience:** $X data not available
4. **Remote vs office percentage:** X% remote, Y% office
5. **Highest paying non-tech industry:** Pharmaceutical Development ($230,000)

**Key insights:**
-The data showed that tech jobs usually pay more than non-tech jobs.
-Cleaning the data helped make the results more correct.
-Checking each column name was important to avoid errors.

**Challenges faced:**
- Challenge 1: I got an error because the column “state” did not exist.
How I solved it: I looked at all column names and used the right one
- Challenge 2: Some data was missing.
How I solved it: I used dropna() to remove rows with missing values.

**What you learned about vibe coding:**
- How to use Pandas to group and find averages.
- How to look at data closely before starting to code.
